In [ ]:
# =====================================================================
# Dynamic-Room Pure Heuristic Optimization
#   Input: ROOM_X, ROOM_Y
#   Method: NSGA-II directly on physics engine (no surrogate)
#   Output: Pareto front + Knee solution
# =====================================================================

import torch, numpy as np, math, time
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.age import AGEMOEA
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.termination import get_termination
from pymoo.optimize import minimize

try:
    device = torch.device('cuda'); torch.zeros(1, device=device)
except:
    device = torch.device('cpu')
print(f'Device: {device}')

# ============================================================
# 1. Room Settings
# ============================================================
ROOM_X, ROOM_Y = 10.0, 10.0   # <-- EDIT THESE
# Self-check: on (10,10), Knee should be ~1.646 m² ± 3%
SELF_CHECK = True  # set False after verified
ROOM_Z = 3.0

# ============================================================
# 2. Physics Engine (per-room KDE)
# ============================================================
xBs,yBs,zBs=10.0,-100.0,-10.0; E=8; dB_s=0.075; lam=0.075
Lp=2;dref=abs(yBs)*1.5;Pd=40.0;Rth=0.2;Nd=-174.0;Bw=20e6;pm=0.2;nu=5
PB=10**(Pd/10)*1e-3;N0f=10**(Nd/10)*1e-3*Bw
Z_HS=[1.5,1.625,1.75,1.875,2.0];N_Z=5;STEP=0.05

def gen_rwp(rx,ry,sim_time,dt=10):
    rng=np.random;rng.seed(777)
    rs=[0.0,rx,0.0,ry];hc=np.array([rx/2,ry/2]);hr=min(rx,ry)/3
    ts=int(sim_time/dt);p_t,p_s=0.6,0.3;tau_h,tau_n=1.5,0.1;v_h,v_n=0.2,1.0;g_h,g_n=0.6,0.2
    def gt():
        t=hc+(rng.rand(2)-0.5)*2*hr if rng.rand()<g_h else np.array([rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])])
        t[0]=np.clip(t[0],rs[0],rs[1]);t[1]=np.clip(t[1],rs[2],rs[3]);return t
    uh=1.5+0.5*rng.rand(nu);pos=np.zeros((nu,ts,2))
    up=np.zeros((nu,2));ur=['normal']*nu;us=[None]*nu;ut_=np.zeros((nu,2));ud=np.zeros((nu,2));usp=np.zeros(nu);uprem=np.zeros(nu)
    for i in range(nu):
        up[i]=[rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])]
        d2h=np.linalg.norm(up[i]-hc);ur[i]='hot' if d2h<=hr else 'normal'
        if rng.rand()<p_t:us[i]='transfer';ut_[i]=gt();dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=(v_h if ur[i]=='hot'else v_n)
        else:us[i]='pause';uprem[i]=rng.exponential(tau_h if ur[i]=='hot'else tau_n);pos[i,0,:]=up[i]
    for step in range(1,ts):
        for i in range(nu):
            if us[i]=='pause':
                uprem[i]-=dt;pos[i,step,:]=up[i]
                if uprem[i]<=0:us[i]='transfer';ut_[i]=gt();dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=(v_h if ur[i]=='hot'else v_n)
            else:
                md=usp[i]*dt;pd=ut_[i]-up[i]
                if np.linalg.norm(pd)<=md:
                    up[i]=ut_[i].copy();d2h=np.linalg.norm(up[i]-hc);ur[i]='hot' if d2h<=hr else 'normal'
                    if rng.rand()<p_s:us[i]='pause';uprem[i]=rng.exponential(tau_h if ur[i]=='hot'else tau_n)
                    else:ut_[i]=gt();dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=(v_h if ur[i]=='hot'else v_n)
                else:up[i]=up[i]+ud[i]*md;pos[i,step,:]=up[i]
    pts=np.zeros((nu*ts,3));idx=0
    for u in range(nu):
        for s in range(ts):pts[idx]=[pos[u,s,0],pos[u,s,1],uh[u]];idx+=1
    return pts

def build_room(rx,ry):
    nx,ny=int(rx/STEP),int(ry/STEP)
    xv=torch.linspace(0,rx,nx,device=device);yv=torch.linspace(0,ry,ny,device=device)
    Xg,Yg=torch.meshgrid(xv,yv,indexing='ij');xyf=torch.stack([Xg.flatten(),Yg.flatten()],dim=1)
    gl=[]
    for zh in Z_HS:gl.append(torch.stack([Xg.flatten(),Yg.flatten(),torch.full_like(Xg.flatten(),zh)],dim=1))
    gl=torch.cat(gl,dim=0);Ng=gl.shape[0]
    et=gen_rwp(rx,ry,100000,10);kde=gaussian_kde(et[:,:2].T,bw_method='scott')
    wxy=kde(xyf.cpu().numpy().T).flatten();wxy=wxy/wxy.sum()
    gw=torch.tensor(np.tile(wxy,N_Z),dtype=torch.float32,device=device);gw=gw/gw.sum()
    np.random.seed(42)
    ps=torch.tensor(2*np.pi*np.random.rand(Ng),dtype=torch.float32,device=device)
    tt=torch.tensor(-np.pi+2*np.pi*np.random.rand(Ng),dtype=torch.float32,device=device)
    eta=torch.tensor(10**((-15+5*np.random.rand(Ng))/10),dtype=torch.float32,device=device)
    na=torch.arange(E,dtype=torch.float32,device=device);dyB=torch.tensor(0.0-yBs,device=device)
    v1c=lam/(4*math.pi);v1pc=-(2*math.pi/lam);oE=1/math.sqrt(E)
    return gl,gw,ps,tt,eta,na,dyB,v1c,v1pc,oE

print(f'Building physics for {ROOM_X}×{ROOM_Y}m room...')
t0=time.time()
PHYS=build_room(ROOM_X,ROOM_Y)
gl,gw,ps,tt,eta,na,dyB,v1c,v1pc,oE=PHYS
print(f'  Done in {time.time()-t0:.0f}s | Grid: {gl.shape[0]} pts | gw sum={gw.sum().item():.4f}')

@torch.no_grad()
def outage(X_4d):
    gl,gw,ps,tt,eta,na,dyB,v1c,v1pc,oE=PHYS
    out=[]
    for i in range(0,len(X_4d),200):
        b=torch.tensor(X_4d[i:i+200],dtype=torch.float32,device=device);bo=torch.zeros(len(b),device=device)
        for j in range(len(b)):
            xc,zc,Lx,Lz=b[j,0],b[j,1],b[j,2],b[j,3]
            xg,yg,zg=gl[:,0],gl[:,1],gl[:,2];Ng=xg.shape[0]
            dx=xc-xBs;dy=dyB;dz=zc-zBs;Rw=torch.sqrt(dx**2+dy**2+dz**2);th=torch.atan2(dy,dx);ph=torch.acos(dz/Rw)
            kx=torch.sin(ph)*torch.cos(th);kz=torch.cos(ph)
            def rd(xs,zs):dd=xs-xBs;dz_=zs-zBs;L=torch.sqrt(dd**2+dy**2+dz_**2);return dd/L,dy/L,dz_/L
            x1,x2=xc-Lx/2,xc-Lx/2;x3,x4=xc+Lx/2,xc+Lx/2;z1,z3=zc-Lz/2,zc-Lz/2;z2,z4=zc+Lz/2,zc+Lz/2
            ux1,_,uz1=rd(x1,z1);ux2,_,uz2=rd(x2,z2);ux3,_,uz3=rd(x3,z3);ux4,_,uz4=rd(x4,z4)
            uma=torch.stack([ux1,ux2,ux3,ux4]);uzm=torch.stack([uz1,uz2,uz3,uz4])
            umin=uma.min(0).values;umax=uma.max(0).values;zmin=uzm.min(0).values;zmax=uzm.max(0).values
            dux=xg-xBs;duy=yg-yBs;duz=zg-zBs;Lu=torch.sqrt(dux**2+duy**2+duz**2);uux=dux/Lu;uuz=duz/Lu
            ix=torch.sigmoid(1000*(uux-umin))*torch.sigmoid(1000*(umax-uux))
            iz=torch.sigmoid(1000*(uuz-zmin))*torch.sigmoid(1000*(zmax-uuz));il=ix*iz
            d2x=xg-xc;d2y=yg;d2z=zg-zc;Ru=torch.sqrt(d2x**2+d2y**2+d2z**2);t1=d2x/Ru;t2=d2z/Ru
            ax=(1-il)*(kx-t1);az=(1-il)*(kz-t2)
            sx=torch.sinc((math.pi/lam)*Lx*ax);sz=torch.sinc((math.pi/lam)*Lz*az)
            p1=(2*math.pi/lam)*dB_s*na.unsqueeze(0)*torch.sin(th);a1=oE*torch.complex(torch.cos(p1),torch.sin(p1))
            v1m=v1c/Rw;v1p=v1pc*Rw;v1=torch.complex(v1m*torch.cos(v1p),v1m*torch.sin(v1p));H1=v1*a1.conj()
            p2=(2*math.pi/lam)*dB_s*na.unsqueeze(0)*torch.sin(tt).unsqueeze(1)
            a2=oE*torch.complex(torch.cos(p2),torch.sin(p2));v2m=eta*(lam/(4*math.pi*dref))
            v2=torch.complex(v2m*torch.cos(ps),v2m*torch.sin(ps));H2=v2.unsqueeze(1)*a2.conj()
            Ht=math.sqrt(E/Lp)*(H1+H2);fm=(Lx*Lz*sx*sz)/(lam*Ru)
            fp=(-(2*math.pi/lam)*(kx*xc+kz*zc)-(math.pi/2));fc=torch.complex(fm*torch.cos(fp),fm*torch.sin(fp))
            He=Ht*fc.unsqueeze(1);Hw=torch.sum(He,dim=1)/math.sqrt(E)
            dp=(torch.abs(Hw)**2)*pm*PB;it=(nu-1)*dp;sr=dp/(it+N0f)
            ab=math.pi/3.0;Ses=torch.zeros(Ng,device=device)
            rn=torch.sqrt((xg-xc)**2+yg**2+(zg-zc)**2+1e-12)
            for a in [0.0,math.pi/2,math.pi,3*math.pi/2]:
                dot=torch.abs((xg-xc)*math.cos(a)+yg*math.sin(a));cp=dot/rn;ph_b=torch.acos(torch.clamp(cp,0,1))
                Se=(math.pi-torch.clamp(2*ab-ph_b,min=0))/math.pi;Ses=Ses+torch.clamp(Se,1/3,1)
            sr_se=((Ses/4)*dp)/((Ses/4)*it+N0f);bits=(torch.log2(1+sr_se)<Rth).float();bo[j]=torch.sum(bits*gw)
        out.append(bo.cpu().numpy());torch.cuda.empty_cache()
    return np.concatenate(out)

# ============================================================
# 3. Optimization
# ============================================================
lb=np.array([0.0,0.0,0.05,0.05]);ub=np.array([ROOM_X,ROOM_Z,ROOM_X,ROOM_Z])

class RoomProblem(ElementwiseProblem):
    def __init__(self):super().__init__(n_var=4,n_obj=2,n_ieq_constr=4,xl=lb,xu=ub)
    def _evaluate(self,x,out,*a,**k):
        xc,zc,Lx,Lz=x
        out["G"]=[Lx/2-xc,xc+Lx/2-ROOM_X,Lz/2-zc,zc+Lz/2-ROOM_Z]
        out["F"]=[Lx*Lz,float(outage(x.reshape(1,-1))[0])]

# Choose algorithm
ALGO_CHOICE = 'NSGA2'  # 'NSGA2' or 'AGE'

if ALGO_CHOICE == 'NSGA2':
    algo=NSGA2(pop_size=300,n_offsprings=150,sampling=FloatRandomSampling(),
        crossover=SBX(prob=0.9,eta=15),mutation=PM(prob=0.25,eta=20),eliminate_duplicates=True)
else:
    algo=AGEMOEA(pop_size=300,n_offsprings=150,sampling=FloatRandomSampling(),
        crossover=SBX(prob=0.9,eta=15),mutation=PM(prob=0.25,eta=20))

# Quick numerical sanity: compare with known 4D reference on 10×10
if abs(ROOM_X-10)<0.1 and abs(ROOM_Y-10)<0.1:
    ref_knee = np.array([[5.264,1.294,9.007,0.183]])  # known 4D knee
    ref_out = outage(ref_knee)[0]
    print(f'Known 4D Knee [{ref_knee[0,0]:.3f},{ref_knee[0,1]:.3f},{ref_knee[0,2]:.3f},{ref_knee[0,3]:.3f}] → outage={ref_out*100:.2f}%')
    print(f'(Expected ~9.97% from 4D engine; large deviation = engine bug)')

print(f'\n{ALGO_CHOICE} on {ROOM_X}×{ROOM_Y}m (pop=300, gen=200)...')
t0=time.time()
res=minimize(RoomProblem(),algo,get_termination('n_gen',200),seed=42,verbose=True)
elapsed=time.time()-t0
print(f'Done: {elapsed:.0f}s, {len(res.F)} Pareto pts')

# ============================================================
# 4. Results
# ============================================================
F=res.F;X=res.X
feas=F[:,1]<=0.10
if feas.any():
    bi=np.argmin(F[feas,0]);ba,bo=F[feas,0][bi],F[feas,1][bi]
    bx=X[np.where(feas)[0][bi]]
    print(f'\nKnee: [{bx[0]:.3f},{bx[1]:.3f},{bx[2]:.3f},{bx[3]:.3f}] area={ba:.4f}m² outage={bo*100:.2f}%')
    if SELF_CHECK and abs(ROOM_X-10)<0.1 and abs(ROOM_Y-10)<0.1:
        ref=1.646; dev=abs(ba-ref)/ref*100
        status='✅ PASS' if dev<5 else '❌ DEVIATION'
        print(f'Self-check: expected ~{ref:.4f}m², got {ba:.4f}m² ({dev:.1f}% off) — {status}')

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5.5))
idx=np.argsort(F[:,1]);Fp=F[idx]
ax1.plot(Fp[:,1]*100,Fp[:,0],'o-',color='#16213e',markersize=2,linewidth=1.2)
ax1.axvline(x=10,color='red',ls='--',alpha=0.5)
if feas.any():ax1.plot(bo*100,ba,'r*',markersize=14,label=f'Knee: {ba:.4f}m² @ {bo*100:.1f}%')
ax1.legend();ax1.set_xlabel('Outage [%]');ax1.set_ylabel('Area [m²]')
ax1.set_title(f'{ALGO_CHOICE} on {ROOM_X}×{ROOM_Y}m, {elapsed:.0f}s');ax1.grid(alpha=0.3)

bx2=bx if feas.any() else np.array([ROOM_X/2,1.5,ROOM_X*0.9,0.2])
ax2.add_patch(Rectangle((bx2[0]-bx2[2]/2,bx2[1]-bx2[3]/2),bx2[2],bx2[3],fill=True,color='#16213e',alpha=0.4,edgecolor='black',lw=2))
ax2.plot(bx2[0],bx2[1],'k*',markersize=12)
ax2.set_xlim(0,ROOM_X);ax2.set_ylim(0,ROOM_Z);ax2.set_aspect('equal')
ax2.set_xlabel('x [m]');ax2.set_ylabel('z [m]');ax2.set_title(f'Knee on {ROOM_X}×{ROOM_Y}m Wall');ax2.grid(alpha=0.3)
plt.tight_layout();plt.savefig('dynamic_pareto.png',dpi=150);plt.show()

from IPython.display import FileLink,display
display(FileLink('dynamic_pareto.png'))